# Kiểm tra kiểu dữ liệu (dtype) trong dữ liệu thô Olist

Task #23 (Story #3): với mỗi bảng, xem dtype thật do `pandas.read_csv` (không truyền `parse_dates`) suy luận, đối chiếu với kiểu logic mong đợi theo `docs/olist_erd.md` (Task #20) — trọng tâm là các cột thời gian (`datetime` trong ERD), vì mặc định `read_csv` luôn đọc chúng thành `object` (chuỗi).

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
csv_files = sorted(RAW_DIR.glob("*.csv"))

dataframes = {f.stem: pd.read_csv(f) for f in csv_files}
print(f"Đã đọc {len(dataframes)} bảng.")

Đã đọc 9 bảng.


In [2]:
for name, df in dataframes.items():
    print(f"=== {name} ===")
    print(df.dtypes.to_string())
    print()

=== olist_customers_dataset ===
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str

=== olist_geolocation_dataset ===
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str

=== olist_order_items_dataset ===
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

=== olist_order_payments_dataset ===
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64

=== olist_order_reviews_dataset ===
review_id                    str
order_id                     str
review_score   

## Đối chiếu cột thời gian (datetime) với ERD

Các cột được ERD mô tả là `datetime` nhưng mặc định `read_csv` sẽ đọc thành `object` (chuỗi), cần `pd.to_datetime` khi xử lý ở Sprint 2.

In [3]:
expected_datetime_cols = {
    "olist_orders_dataset": [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "olist_order_items_dataset": ["shipping_limit_date"],
    "olist_order_reviews_dataset": ["review_creation_date", "review_answer_timestamp"],
}

rows = []
for name, cols in expected_datetime_cols.items():
    df = dataframes[name]
    for col in cols:
        actual = str(df[col].dtype)
        rows.append({
            "bảng": name,
            "cột": col,
            "dtype mong đợi": "datetime",
            "dtype thực tế": actual,
            "khớp?": actual.startswith("datetime"),
        })

datetime_check = pd.DataFrame(rows)
datetime_check

,bảng,cột,dtype mong đợi,dtype thực tế,khớp?
0,olist_orders_dataset,order_purchase_timestamp,datetime,str,False
1,olist_orders_dataset,order_approved_at,datetime,str,False
2,olist_orders_dataset,order_delivered_carrier_date,datetime,str,False
3,olist_orders_dataset,order_delivered_customer_date,datetime,str,False
4,olist_orders_dataset,order_estimated_delivery_date,datetime,str,False
5,olist_order_items_dataset,shipping_limit_date,datetime,str,False
6,olist_order_reviews_dataset,review_creation_date,datetime,str,False
7,olist_order_reviews_dataset,review_answer_timestamp,datetime,str,False


## Đối chiếu cột số (numeric)

Các cột đo lường/giá trị (giá, phí, số lượng, điểm đánh giá, kích thước sản phẩm) cần là kiểu số (`int64`/`float64`), không phải `object`.

In [4]:
expected_numeric_cols = {
    "olist_order_items_dataset": ["price", "freight_value"],
    "olist_order_payments_dataset": ["payment_sequential", "payment_installments", "payment_value"],
    "olist_order_reviews_dataset": ["review_score"],
    "olist_products_dataset": [
        "product_name_lenght", "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    ],
    "olist_geolocation_dataset": ["geolocation_lat", "geolocation_lng"],
}

rows = []
for name, cols in expected_numeric_cols.items():
    df = dataframes[name]
    for col in cols:
        actual = str(df[col].dtype)
        rows.append({
            "bảng": name,
            "cột": col,
            "dtype mong đợi": "số (int/float)",
            "dtype thực tế": actual,
            "khớp?": actual.startswith(("int", "float")),
        })

numeric_check = pd.DataFrame(rows)
numeric_check

,bảng,cột,dtype mong đợi,dtype thực tế,khớp?
0,olist_order_items_dataset,price,số (int/float),float64,True
1,olist_order_items_dataset,freight_value,số (int/float),float64,True
2,olist_order_payments_dataset,payment_sequential,số (int/float),int64,True
3,olist_order_payments_dataset,payment_installments,số (int/float),int64,True
4,olist_order_payments_dataset,payment_value,số (int/float),float64,True
5,olist_order_reviews_dataset,review_score,số (int/float),int64,True
6,olist_products_dataset,product_name_lenght,số (int/float),float64,True
7,olist_products_dataset,product_description_lenght,số (int/float),float64,True
8,olist_products_dataset,product_photos_qty,số (int/float),float64,True
9,olist_products_dataset,product_weight_g,số (int/float),float64,True


## Tổng hợp cột sai kiểu dữ liệu

In [5]:
all_checks = pd.concat([datetime_check, numeric_check], ignore_index=True)
mismatches = all_checks[~all_checks["khớp?"]]
print(f"{len(mismatches)}/{len(all_checks)} cột sai kiểu dữ liệu so với kỳ vọng.")
mismatches

8/23 cột sai kiểu dữ liệu so với kỳ vọng.


,bảng,cột,dtype mong đợi,dtype thực tế,khớp?
0,olist_orders_dataset,order_purchase_timestamp,datetime,str,False
1,olist_orders_dataset,order_approved_at,datetime,str,False
2,olist_orders_dataset,order_delivered_carrier_date,datetime,str,False
3,olist_orders_dataset,order_delivered_customer_date,datetime,str,False
4,olist_orders_dataset,order_estimated_delivery_date,datetime,str,False
5,olist_order_items_dataset,shipping_limit_date,datetime,str,False
6,olist_order_reviews_dataset,review_creation_date,datetime,str,False
7,olist_order_reviews_dataset,review_answer_timestamp,datetime,str,False
